In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
run_bidirectional_4244.py
-------------------------
指定フォルダ配下に既にある CSV 群から、4.2.4.4 の
「順方向（OH→FP）× 逆方向（FP→OH）」を結合・相関・図化する
オールインワン実行スクリプト（Jupyter/サーバでも安定動作）。

【既定ベース（変更可）】
  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022

【想定配置例】
  <base>/
    ├─ reverse/analysis_csv/
    │    ├─ Table_4.2.4.4R_fragment-to-OHmajor_*.csv
    │    ├─ Table_4.2.4.4R_fragment-pair_cohesion_*.csv
    │    └─ Table_4.2.4.4R_FPcluster_cohesion_*.csv
    ├─ main_text/（またはルート直下 など）
    │    └─ Table_4.2.4.4_OHFP-contrast_summary_k-le30*.csv
    └─ paper_4.2.4.4_OHFP/
         └─ bidirectional_summary_YYYYMMDD_HHMM/  ← 本スクリプトの出力既定

【出力（既定）】
  <base>/paper_4.2.4.4_OHFP/bidirectional_summary_YYYYMMDD_HHMM/
    ├─ Table_4.2.4.4_bidirectional_joined.csv
    ├─ Table_4.2.4.4_bidirectional_correlations.csv
    ├─ Fig_4.2.4.4_bidirectional_*.png / .pdf
    └─ README_bidirectional_summary.json

【実行例】
  # そのまま（既定ベース）
  python run_bidirectional_4244.py

  # 別ベースを使う
  python run_bidirectional_4244.py --base /path/to/another/root

  # 出力先を固定（日時フォルダにせず）
  python run_bidirectional_4244.py --no-timestamp --out /path/to/outdir

  # 逆方向 analysis_csv と 順方向CSV を明示指定
  python run_bidirectional_4244.py \
      --reverse-dir /path/to/reverse/analysis_csv \
      --forward-file /path/to/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv \
      --save-zip
"""

from __future__ import annotations
import argparse, sys, os, re, json, zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # GUI不要環境向け
import matplotlib.pyplot as plt

# ------------------------------
# 既定のベースフォルダ
# ------------------------------
DEFAULT_BASE = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022")

# ------------------------------
# 探索ユーティリティ（パターン）
# ------------------------------
FWD_PAT = re.compile(r"^Table_4\.2\.4\.4_OHFP-contrast_summary_k-le30.*\.csv$")
REV_PAT = re.compile(
    r"^Table_4\.2\.4\.4R_(fragment-to-OHmajor|fragment-pair_cohesion|FPcluster_cohesion)_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)\.csv$"
)

def find_forward_csv(base: Path) -> Path | None:
    cands = [p for p in base.rglob("*.csv") if FWD_PAT.match(p.name)]
    if not cands:
        return None
    return sorted(cands, key=os.path.getmtime)[-1]  # 最終更新が新しいもの

def find_reverse_analysis_dir(base: Path) -> Path | None:
    """
    base 配下から reverse/analysis_csv を優先的に見つける。
    無い場合は REV_PAT にマッチする CSV が最も多いフォルダを推測。
    """
    for p in base.rglob("analysis_csv"):
        files = list(p.glob("Table_4.2.4.4R_*.csv"))
        if sum(1 for f in files if REV_PAT.match(f.name)) >= 3:
            return p
    best_dir, best_cnt = None, 0
    for p in base.rglob("*.csv"):
        if REV_PAT.match(p.name):
            cnt = len([f for f in p.parent.glob("Table_4.2.4.4R_*.csv") if REV_PAT.match(f.name)])
            if cnt > best_cnt:
                best_dir, best_cnt = p.parent, cnt
    return best_dir

# ------------------------------
# 逆方向サマリーの再構築（集計）
# ------------------------------
def rebuild_reverse_summary(analysis_csv_dir: Path) -> pd.DataFrame:
    pat = re.compile(
        r"Table_4\.2\.4\.4R_(?P<kind>fragment-to-OHmajor|fragment-pair_cohesion|FPcluster_cohesion)"
        r"_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)\.csv$"
    )
    by_key: dict[tuple[str,str], dict] = {}
    for f in analysis_csv_dir.glob("Table_4.2.4.4R_*.csv"):
        m = pat.match(f.name)
        if not m:
            continue
        kind = m["kind"]; mode = m["mode"]; index = m["index"]
        key = (mode, index)
        by_key.setdefault(key, {"mode": mode, "index": index})
        df = pd.read_csv(f)

        if kind == "fragment-to-OHmajor":
            pur = pd.to_numeric(df.get("OH_purity"), errors="coerce")
            ent = pd.to_numeric(df.get("OH_entropy"), errors="coerce")
            by_key[key]["mean_OH_purity"]  = float(pur.dropna().mean()) if pur.notna().any() else np.nan
            by_key[key]["mean_OH_entropy"] = float(ent.dropna().mean()) if ent.notna().any() else np.nan
            by_key[key]["n_fragments"]     = int(df.shape[0])

        elif kind == "fragment-pair_cohesion":
            ms = df.get("OH_major_same")
            if ms is not None:
                ms2 = ms.copy()
                if ms2.dtype == "O":
                    ms2 = ms2.astype(str).str.lower().map({"true": True, "false": False})
                by_key[key]["pair_OH_major_same_rate"] = float(pd.to_numeric(ms2, errors="coerce").mean())
            cs = pd.to_numeric(df.get("OH_cosine_similarity"), errors="coerce")
            js = pd.to_numeric(df.get("OH_JS_divergence"),   errors="coerce")
            by_key[key]["pair_mean_cosine_similarity"] = float(cs.dropna().mean()) if cs.notna().any() else np.nan
            by_key[key]["pair_mean_JS_divergence"]     = float(js.dropna().mean()) if js.notna().any() else np.nan
            # 方向を合わせるための反転スコア（高いほど良い）
            if "JS_divergence_rev" not in by_key[key]:
                by_key[key]["JS_divergence_rev"] = np.nan  # 後で補う

        elif kind == "FPcluster_cohesion":
            cr = pd.to_numeric(df.get("cohesive_ratio"), errors="coerce")
            by_key[key]["FPcluster_median_cohesive_ratio"] = float(cr.dropna().median()) if cr.notna().any() else np.nan

    # JS 反転スコアを補完（0-1範囲であれば 1 - JS、範囲外なら min-max → 1 - x）
    for key, d in by_key.items():
        js = d.get("pair_mean_JS_divergence", np.nan)
        if np.isfinite(js):
            if 0.0 <= js <= 1.0:
                d["JS_divergence_rev"] = 1.0 - js
            else:
                # 単一集計では min-max の参照が無いので、そのまま NA にせず、生値から経験的に 1 - (js - floor)/range は不可。
                # 実運用では「joined後の列」で min-max 正規化するのが安全。ここではそのままにしておく。
                pass

    return pd.DataFrame(sorted(by_key.values(), key=lambda d:(d["mode"], d["index"])))

# ------------------------------
# 順方向の横持ち化 + 平均列
# ------------------------------
def make_forward_wide(fwd_csv: Path) -> pd.DataFrame:
    fwd = pd.read_csv(fwd_csv)
    keep = ["mode","index","Dataset","ARI","NMI","AMI","kA","kB","n"]
    for c in keep:
        if c not in fwd.columns:
            fwd[c] = np.nan
    w = (fwd[keep]
         .pivot_table(index=["mode","index"],
                      columns="Dataset",
                      values=["ARI","NMI","AMI","kA","kB","n"],
                      aggfunc="first"))
    w.columns = [f"{a}_{b}" for a,b in w.columns]
    w = w.reset_index()
    # 列名末尾を大文字化
    w = w.rename(columns={c: c.replace("_oh","_OH").replace("_fp","_FP") for c in w.columns})

    def mean2(a, b):
        a = pd.to_numeric(a, errors="coerce")
        b = pd.to_numeric(b, errors="coerce")
        return pd.concat([a,b], axis=1).mean(axis=1, skipna=True)

    if {"ARI_OH","ARI_FP"}.issubset(w.columns): w["ARI_mean"] = mean2(w["ARI_OH"], w["ARI_FP"])
    if {"NMI_OH","NMI_FP"}.issubset(w.columns): w["NMI_mean"] = mean2(w["NMI_OH"], w["NMI_FP"])
    if {"AMI_OH","AMI_FP"}.issubset(w.columns): w["AMI_mean"] = mean2(w["AMI_OH"], w["AMI_FP"])
    return w

# ------------------------------
# 相関計算（Pearson/Spearman）
# ------------------------------
def compute_correlations(df_merged: pd.DataFrame) -> pd.DataFrame:
    import scipy.stats as stats
    rev_metrics = [
        "pair_OH_major_same_rate",
        "pair_mean_cosine_similarity",
        "pair_mean_JS_divergence",
        "JS_divergence_rev",
        "FPcluster_median_cohesive_ratio",
        "mean_OH_purity", "mean_OH_entropy"
    ]
    f_metrics = [c for c in ["ARI_mean","NMI_mean","AMI_mean"] if c in df_merged.columns]
    rows=[]
    for fm in f_metrics:
        for rm in rev_metrics:
            if rm not in df_merged.columns:
                continue
            x = pd.to_numeric(df_merged[fm], errors="coerce")
            y = pd.to_numeric(df_merged[rm], errors="coerce")
            m = x.notna() & y.notna()
            if m.sum() >= 3:
                pr = stats.pearsonr(x[m], y[m])
                sr = stats.spearmanr(x[m], y[m])
                rows.append({
                    "forward_metric": fm, "reverse_metric": rm, "n": int(m.sum()),
                    "pearson_r": float(pr.statistic), "pearson_p": float(pr.pvalue),
                    "spearman_rho": float(sr.statistic), "spearman_p": float(sr.pvalue)
                })
            else:
                rows.append({
                    "forward_metric": fm, "reverse_metric": rm, "n": int(m.sum()),
                    "pearson_r": np.nan, "pearson_p": np.nan,
                    "spearman_rho": np.nan, "spearman_p": np.nan
                })
    return pd.DataFrame(rows)

# ------------------------------
# 図（シンプル散布・回帰線）
# ------------------------------
def _scatter(ax, x, y, title=""):
    ax.scatter(x, y, alpha=0.9)
    if len(x) >= 2 and len(y) >= 2:
        coef = np.polyfit(x, y, 1)
        xs = np.linspace(float(np.min(x)), float(np.max(x)), 100)
        ys = np.polyval(coef, xs)
        ax.plot(xs, ys)
    ax.set_title(title)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)

def save_core_scatter(df, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    pairs = [
        ("ARI_mean","pair_OH_major_same_rate","ARI_mean vs pair_OH_major_same_rate"),
        ("ARI_mean","pair_mean_cosine_similarity","ARI_mean vs pair_mean_cosine_similarity"),
        ("ARI_mean","FPcluster_median_cohesive_ratio","ARI_mean vs FPcluster_median_cohesive_ratio"),
        ("ARI_mean","JS_divergence_rev","ARI_mean vs JS_divergence_rev"),
    ]
    for xcol, ycol, ttl in pairs:
        if xcol not in df.columns or ycol not in df.columns:
            continue
        x = pd.to_numeric(df[xcol], errors="coerce")
        y = pd.to_numeric(df[ycol], errors="coerce")
        m = x.notna() & y.notna()
        if m.sum() < 2:
            continue
        fig, ax = plt.subplots(figsize=(6,4), dpi=300)
        _scatter(ax, x[m].values, y[m].values, ttl)
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)  # 英語ラベル固定
        fig.tight_layout()
        fig.savefig(out_dir / f"Fig_4.2.4.4_bidirectional_{xcol}_vs_{ycol}.png", bbox_inches="tight")
        fig.savefig(out_dir / f"Fig_4.2.4.4_bidirectional_{xcol}_vs_{ycol}.pdf", bbox_inches="tight")
        plt.close(fig)

# ------------------------------
# 出力ディレクトリの決定
# ------------------------------
def resolve_outdir(base: Path, user_out: str | None, use_timestamp: bool) -> Path:
    if user_out:
        out_root = Path(user_out).resolve()
    else:
        root = base / "paper_4.2.4.4_OHFP"
        if use_timestamp:
            ts = datetime.now().strftime("%Y%m%d_%H%M")
            out_root = root / f"bidirectional_summary_{ts}"
        else:
            out_root = root / "bidirectional_summary"
    out_root.mkdir(parents=True, exist_ok=True)
    return out_root

# ------------------------------
# メイン
# ------------------------------
def main():
    ap = argparse.ArgumentParser(description="4.2.4.4 bidirectional builder (folder-based)")
    ap.add_argument("--base", type=str, default=str(DEFAULT_BASE),
                    help="探索の起点フォルダ（省略時: 既定ベース）")
    ap.add_argument("--reverse-dir", type=str, default=None,
                    help="逆方向 per-condition CSV (analysis_csv) を直接指定")
    ap.add_argument("--forward-file", type=str, default=None,
                    help="順方向本文表 CSV（Table_4.2.4.4_OHFP-contrast_summary_k-le30*.csv）を直接指定")
    ap.add_argument("--out", type=str, default=None,
                    help="出力フォルダ。未指定時は base/paper_4.2.4.4_OHFP/bidirectional_summary[_YYYYMMDD_HHMM]")
    ap.add_argument("--no-timestamp", action="store_true",
                    help="日時付きフォルダを使わず、固定 bidirectional_summary に出力する")
    ap.add_argument("--save-zip", action="store_true", help="出力一式を zip 化する")
    args, _ = ap.parse_known_args()  # Jupyter の -f 対策

    base = Path(args.base).resolve()
    if not base.exists():
        raise FileNotFoundError(f"base が存在しません: {base}")

    out_root = resolve_outdir(base, args.out, use_timestamp=(not args.no_timestamp))
    print(f"[OUT] {out_root}")

    # 1) 逆方向 analysis_csv
    if args.reverse_dir:
        rev_dir = Path(args.reverse_dir).resolve()
        if not rev_dir.exists():
            raise FileNotFoundError(f"--reverse-dir が存在しません: {rev_dir}")
    else:
        rev_dir = find_reverse_analysis_dir(base)
        if rev_dir is None:
            raise FileNotFoundError("逆方向の analysis_csv フォルダが見つかりませんでした。--reverse-dir で明示指定してください。")
    print(f"[REV] analysis_csv directory: {rev_dir}")

    # 2) 順方向（本文表）CSV
    if args.forward_file:
        fwd_csv = Path(args.forward_file).resolve()
        if not fwd_csv.exists():
            raise FileNotFoundError(f"--forward-file が見つかりません: {fwd_csv}")
    else:
        fwd_csv = find_forward_csv(base)
        if fwd_csv is None:
            raise FileNotFoundError("順方向 CSV が見つかりませんでした。--forward-file で明示指定してください。")
    print(f"[FWD] summary file: {fwd_csv}")

    # 3) 逆方向サマリー再構築
    rev_summary = rebuild_reverse_summary(rev_dir)
    # JS 反転列が欠けていれば、結合後に補う（min-max→反転）ためフラグ保持
    need_js_rev_post = "JS_divergence_rev" not in rev_summary.columns or rev_summary["JS_divergence_rev"].isna().all()

    # 4) 順方向 横持ち化
    fwd_wide = make_forward_wide(fwd_csv)

    # 5) マージ
    merged = (fwd_wide.merge(rev_summary, on=["mode","index"], how="outer")
                        .sort_values(["mode","index"]).reset_index(drop=True))

    # mode/index 欠損の安全化
    for c in ["mode","index"]:
        if c not in merged.columns:
            merged[c] = "NA"
        merged[c] = merged[c].fillna("NA")

    # JS 反転（後補）：pair_mean_JS_divergence が 0-1 外なら min-max 正規化→ 1 - x
    if need_js_rev_post and "pair_mean_JS_divergence" in merged.columns:
        js = pd.to_numeric(merged["pair_mean_JS_divergence"], errors="coerce")
        if js.notna().sum() > 0:
            js_min, js_max = js.min(), js.max()
            if np.isfinite(js_min) and np.isfinite(js_max) and js_max > js_min:
                js01 = (js - js_min) / (js_max - js_min)
                merged["JS_divergence_rev"] = 1.0 - js01
            else:
                merged["JS_divergence_rev"] = np.nan

    # 保存：joined
    joined_csv = out_root / "Table_4.2.4.4_bidirectional_joined.csv"
    merged.to_csv(joined_csv, index=False, encoding="utf-8-sig")
    print("[SAVE]", joined_csv)

    # 6) 相関 & 図
    corr = compute_correlations(merged)
    corr_csv = out_root / "Table_4.2.4.4_bidirectional_correlations.csv"
    corr.to_csv(corr_csv, index=False, encoding="utf-8-sig")
    print("[SAVE]", corr_csv)

    figs_dir = out_root  # 同階層に保存（PNG/PDF）
    save_core_scatter(merged, figs_dir)

    # 7) README 生成
    readme = {
        "base": str(base),
        "reverse_analysis_dir": str(rev_dir),
        "forward_csv": str(fwd_csv),
        "outputs": {
            "joined": str(joined_csv),
            "correlations": str(corr_csv),
            "figs": sorted([p.name for p in figs_dir.glob("Fig_4.2.4.4_bidirectional_*.png")]),
        }
    }
    (out_root / "README_bidirectional_summary.json").write_text(
        json.dumps(readme, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("[SAVE]", out_root / "README_bidirectional_summary.json")

    # 8) zip（任意）
    if args.save_zip:
        zip_path = out_root.with_suffix(".zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for p in out_root.rglob("*"):
                z.write(p, p.relative_to(out_root.parent))
        print("[ZIP ]", zip_path)

    print("\n✅ Done. outputs ->", out_root)

if __name__ == "__main__":
    main()

[OUT] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/paper_4.2.4.4_OHFP/bidirectional_summary_20251023_1145
[REV] analysis_csv directory: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/reverse/analysis_csv
[FWD] summary file: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/main_text/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/paper_4.2.4.4_OHFP/bidirectional_summary_20251023_1145/Table_4.2.4.4_bidirectional_joined.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/paper_4.2.4.4_OHFP/bidirectional_summary_20251023_1145/Table_4.2.4.4_bidirectional_correlations.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/paper_4.2.4.4_OHFP/bidirectional_summary_20251023_1145/README_bidire

In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
paper_4.2.4.4_build_all_FPLabel.py (k-mode 拡張版)
-------------------------------------------------
4.2.4.4節 一括再解析 + 3パターンの k 取り扱いを CLI で切替可能:

  --k-mode legacy   : forward= k≤30, reverse= all-k（従来）
  --k-mode allk     : forward= all-k, reverse= all-k
  --k-mode unify30  : forward= k≤30, reverse= k≤30（両方向を統一）

出力構成は従来通り：
  ROOT/paper_4.2.4.4_OHFP/{main_text, appendix, reverse/*, bidirectional_summary}

※ reverse 側のサマリーに kOH/kFP/k_max を新たに付与。
※ bidirectional_summary（レガシー棒グラフ）は k-mode に応じて forward/reverse を選択。
※ joined→相関→図→統計 ブロックでも選ばれた forward/reverse を用いる。
"""

from __future__ import annotations
from pathlib import Path
import os, re, glob, json, argparse, zipfile, math, warnings
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.patches import Patch

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from scipy.spatial.distance import cosine
from scipy import stats

# ====== 依存（任意）：statsmodels があれば BH に使用、なければ手実装 ======
try:
    from statsmodels.stats.multitest import multipletests
    _HAS_STATSMODELS = True
except Exception:
    _HAS_STATSMODELS = False

# ========= ユーザー設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")

FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"

# signless 側（OH/FP）
SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"  # 例
SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}

# features（材料列のみ）
FEATURES_OH = ROOT / "features_OH_onlyMat.csv"
FEATURES_FP = ROOT / "features_FP_onlyMat.csv"

# 出力ルート
OUT_OHFP_ROOT     = ROOT / "paper_4.2.4.4_OHFP"
OUT_MAIN          = OUT_OHFP_ROOT / "main_text"
OUT_APPENDIX      = OUT_OHFP_ROOT / "appendix"
OUT_REVERSE_ROOT  = OUT_OHFP_ROOT / "reverse"
OUT_R_MAIN        = OUT_REVERSE_ROOT / "main_text"
OUT_R_APPENDIX    = OUT_REVERSE_ROOT / "appendix"
OUT_R_CSV         = OUT_REVERSE_ROOT / "analysis_csv"
OUT_BIDIR         = OUT_OHFP_ROOT / "bidirectional_summary"

for d in [OUT_MAIN, OUT_APPENDIX, OUT_R_MAIN, OUT_R_APPENDIX, OUT_R_CSV, OUT_BIDIR]:
    d.mkdir(parents=True, exist_ok=True)

# 対象の mode/index
DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
MAIN_PAIRS = {("cumeig","gap"), ("cumeig","db")}  # 本文優先

# 既定（legacy を再現するためのデフォルト）
DEFAULT_K_MODE = "legacy"   # ["legacy","allk","unify30"]
K_MAX_KEEP_DEFAULT = 30     # legacy / unify30 用のしきい値

DPI = 300
np.random.seed(4244)

# ========= フォント =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo","Hiragino Sans","Meiryo UI"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try: have.add(fm.FontProperties(fname=p).get_name())
        except: pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams.update({"font.size":10, "axes.titlesize":12, "axes.labelsize":10, "legend.fontsize":9})

# ========= 汎用IO =========
def read_csv_quiet(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists(): return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None

def load_features_bin(path: Path) -> pd.DataFrame:
    df = read_csv_quiet(path)
    if df is None: raise FileNotFoundError(path)
    if df.columns[0].lower()!="sample_id":
        df = df.rename(columns={df.columns[0]:"sample_id"})
    X = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return (X!=0).astype(int)

# ========= FP名 → 表示名（出力用） =========
FP_ROLE_MAP = {None:"in", "1":"ip1", "2":"ip2", "3":"n", "4":"p1", "5":"p2"}
FP_PAT = re.compile(r"^X(?P<num>\d+)(?:\.(?P<suf>[1-5]))?$", flags=re.IGNORECASE)
def parse_fp_var(fp_var: str) -> tuple[str|None, str|None]:
    s = str(fp_var).strip()
    m = FP_PAT.match(s)
    if not m:
        return None, None
    num = m.group("num")
    suf = m.group("suf")
    role = FP_ROLE_MAP.get(suf, None if suf is not None else "in")
    return num, role

def fp_var_display_name(fp_var: str) -> str:
    num, role = parse_fp_var(fp_var)
    if num is None or role is None:
        return fp_var
    return f"X{num}[{role}]"

# ========= OH→FP：new vs signless の一致度 =========
def collect_assign_labels(figs_dir: Path, ds: str) -> pd.DataFrame | None:
    pat = re.compile(rf"^ClusterAssign_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)_{ds}_.*\.csv$")
    files = [p for p in figs_dir.glob("ClusterAssign_*.csv") if pat.match(p.name)]
    parts=[]
    for f in files:
        m = re.match(r"ClusterAssign_(top3|cumeig)_([A-Za-z]+)_", f.name)
        if not m: continue
        mode, index = m.groups()
        df = read_csv_quiet(f)
        if df is None: continue
        if {"Variable","Cluster"}.issubset(df.columns):
            ids = df["Variable"].astype(str).values
            cl  = df["Cluster"].values
        elif df.shape[1]==1:
            ids = df.index.astype(str).values
            cl  = df.iloc[:,0].values
        else:
            continue
        codes = pd.Categorical(cl).codes
        codes = pd.Series(codes).replace({-1: np.nan}).values
        parts.append(pd.DataFrame({"id": ids, f"{mode}_{index}": codes}))
    if not parts: return None
    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on="id", how="outer")
    return out.drop_duplicates(subset=["id"])

def load_signless_labels(signless_dir: Path) -> pd.DataFrame | None:
    cands = sorted(signless_dir.glob("cluster_labels_*.csv"), key=os.path.getmtime, reverse=True)
    if not cands: return None
    df = read_csv_quiet(cands[0])
    if df is None or "id" not in df.columns: return None
    keep = [c for c in df.columns if str(c).startswith("linear_")]
    if not keep: return None
    out = df[["id"]+keep].copy()
    out.columns = ["id"] + [c.replace("linear_","",1) for c in keep]
    return out

def pairwise_scores(dfA: pd.DataFrame|None, dfB: pd.DataFrame|None) -> pd.DataFrame:
    cols = ["condition","n","kA","kB","ARI","NMI","AMI"]
    if dfA is None or dfB is None: return pd.DataFrame(columns=cols)
    colsA = [c for c in dfA.columns if c!="id"]
    colsB = [c for c in dfB.columns if c!="id"]
    common = sorted(set(colsA).intersection(colsB))
    if not common: return pd.DataFrame(columns=cols)
    merged = dfA.merge(dfB, on="id", suffixes=(".A",".B"))
    rows=[]
    for cn in common:
        a = merged[f"{cn}.A"].values
        b = merged[f"{cn}.B"].values
        mask = ~pd.isna(a) & ~pd.isna(b)
        if mask.sum() < 2: continue
        a = a[mask].astype(int); b = b[mask].astype(int)
        rows.append([
            cn, int(mask.sum()),
            int(len(np.unique(a))), int(len(np.unique(b))),
            float(adjusted_rand_score(a,b)),
            float(normalized_mutual_info_score(a,b)),
            float(adjusted_mutual_info_score(a,b)),
        ])
    return pd.DataFrame(rows, columns=cols)

def lighten(color, factor=0.5):
    r,g,b = matplotlib.colors.to_rgb(color)
    return (1 - factor) + factor*r, (1 - factor) + factor*g, (1 - factor) + factor*b

def save_bar_contrast_with_k(df_long: pd.DataFrame, T_meta: pd.DataFrame, out_dir: Path, title: str, fname_core: str, fade_outside: bool):
    set_font()
    INDEX_LIST = ["silhouette","dunn","gap","ch","db","ptbiserial"]
    DATASETS = ["OH","FP"]

    df = df_long.copy()
    df["mode"]  = pd.Categorical(df["mode"], categories=["top3","cumeig"], ordered=True)
    df["index"] = pd.Categorical(df["index"], categories=INDEX_LIST, ordered=True)
    df = df.sort_values(["index","mode","Dataset"])

    x_labels = [f"{m}\n{ix}" for m in ["top3","cumeig"] for ix in INDEX_LIST]
    grid=[]
    for m in ["top3","cumeig"]:
        for ix in INDEX_LIST:
            sl = df[(df["mode"]==m) & (df["index"]==ix)]
            grid.append(sl.set_index("Dataset")["ARI"].reindex(DATASETS))
    X = pd.concat(grid, axis=1).T
    X.index = x_labels

    M = T_meta.copy()
    M["row_key"] = M["mode"] + "\n" + M["index"]
    M["is_target"] = (M["kA"]<=K_MAX_KEEP_DEFAULT) & (M["kB"]<=K_MAX_KEEP_DEFAULT)
    META = (M.set_index(["row_key","Dataset"])[["kA","kB","is_target"]]
              .reindex(pd.MultiIndex.from_product([x_labels, DATASETS], names=["row_key","Dataset"])))

    color_OH = "#1f77b4"; color_FP="#ff7f0e"
    fade_OH  = lighten(color_OH, 0.7); fade_FP = lighten(color_FP, 0.7)

    fig, ax = plt.subplots(figsize=(12,6), dpi=DPI)
    width = 0.38; x = np.arange(X.shape[0])

    for i, ds in enumerate(DATASETS):
        base_color = color_OH if ds=="OH" else color_FP
        base_fade  = fade_OH  if ds=="OH" else fade_FP
        xs = x + (i-0.5)*width
        vals = X[ds].values
        for xi, val, row_key in zip(xs, vals, X.index):
            if (row_key, ds) in META.index:
                kA = META.loc[(row_key, ds), "kA"]
                kB = META.loc[(row_key, ds), "kB"]
                is_t = bool(META.loc[(row_key, ds), "is_target"])
            else:
                kA = np.nan; kB = np.nan; is_t=False
            fc = base_color if (not fade_outside or is_t) else base_fade
            alpha = 0.95 if (not fade_outside or is_t) else 0.5
            ax.bar(xi, val, width, color=fc, alpha=alpha, edgecolor="none")
            if not np.isnan(val):
                label = f"{val:.2f}\n({int(kA) if not np.isnan(kA) else '-'}" \
                        f"/{int(kB) if not np.isnan(kB) else '-'})"
                ax.text(xi, val + 0.015, label, ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x); ax.set_xticklabels(X.index, rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("ARI")
    ax.set_title(title)
    legend_elems = [
        Patch(facecolor=color_OH, edgecolor="none", alpha=0.95, label="OH (k≤30)"),
        Patch(facecolor=color_FP, edgecolor="none", alpha=0.95, label="FP (k≤30)")
    ]
    if fade_outside:
        legend_elems += [
            Patch(facecolor=fade_OH, edgecolor="none", alpha=0.50, label="OH (k>30)"),
            Patch(facecolor=fade_FP, edgecolor="none", alpha=0.50, label="FP (k>30)")
        ]
    ax.legend(handles=legend_elems, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()

    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{fname_core}.png", bbox_inches="tight")
    fig.savefig(out_dir / f"{fname_core}.pdf", bbox_inches="tight")
    plt.close(fig)

# ===== forward 側：本文表（k≤30 と all-k の両方を作れるように） =====
def build_ohfp_contrast_tables() -> dict:
    """
    戻り値：{
      "allk": DataFrame,
      "k30" : DataFrame,
      "allk_path": Path,
      "k30_path" : Path
    }
    """
    all_tbl_allK=[]; all_for_contrast_allK=[]
    all_tbl_k30=[];  all_for_contrast_k30=[]

    for ds, figs in [("OH",FIGS_OH), ("FP",FIGS_FP)]:
        A = collect_assign_labels(figs, ds)
        S = load_signless_labels(SIGNLESS_DIR[ds])
        T = pairwise_scores(A, S)
        if T.empty:
            print(f"[WARN] {ds}: no common conditions")
            continue
        T["mode"]    = T["condition"].str.replace(r"_(.*)$", "", regex=True)
        T["index"]   = T["condition"].str.replace(r"^(.*)_", "", regex=True)
        T["Dataset"] = ds
        T["k_max"]   = T[["kA","kB"]].max(axis=1)

        # all-k
        all_tbl_allK.append(T[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
        all_for_contrast_allK.append(T[["Dataset","mode","index","ARI","kA","kB","k_max"]])

        # k≤30
        T30 = T[(T["k_max"].isna()) | (T["k_max"]<=K_MAX_KEEP_DEFAULT)].copy()
        if len(T30):
            all_tbl_k30.append(T30[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
            all_for_contrast_k30.append(T30[["Dataset","mode","index","ARI","kA","kB","k_max"]])

    out = {}
    if all_tbl_allK:
        T_allk = pd.concat(all_tbl_allK, ignore_index=True)
        p_allk = OUT_APPENDIX / "Appendix_Table_allK_OHFP-contrast_summary.csv"
        T_allk.sort_values(["Dataset","mode","index"]).to_csv(p_allk, index=False, encoding="utf-8-sig")
        out["allk"] = T_allk; out["allk_path"] = p_allk
        print(f"[SAVE] {p_allk}")
    else:
        out["allk"] = pd.DataFrame(); out["allk_path"] = OUT_APPENDIX / "Appendix_Table_allK_OHFP-contrast_summary.csv"

    if all_tbl_k30:
        T_k30 = pd.concat(all_tbl_k30, ignore_index=True)
        p_k30 = OUT_MAIN / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"
        T_k30.sort_values(["Dataset","mode","index"]).to_csv(p_k30, index=False, encoding="utf-8-sig")
        out["k30"] = T_k30; out["k30_path"] = p_k30
        print(f"[SAVE] {p_k30}")

        # 本文の棒図（従来）
        for mode,index in MAIN_PAIRS:
            sub = pd.concat(all_for_contrast_k30, ignore_index=True)
            sub = sub[(sub["mode"]==mode)&(sub["index"]==index)]
            out_dir = OUT_MAIN / f"FigSet_4.2.4.4_{mode}_{index}"
            save_bar_contrast_with_k(
                df_long=sub,
                T_meta=T_k30[(T_k30["mode"]==mode)&(T_k30["index"]==index)][["Dataset","mode","index","kA","kB","k_max"]],
                out_dir=out_dir,
                title=f"Fig. 4.2.4.4: OH vs FP (ARI of new vs signless, {mode}×{index}, k≤{K_MAX_KEEP_DEFAULT})",
                fname_core="Fig_4.2.4.4_OHFP-contrast_ARI-bars",
                fade_outside=False
            )
    else:
        out["k30"] = pd.DataFrame(); out["k30_path"] = OUT_MAIN / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"

    # 付録 all-k 図（全体像）
    if all_for_contrast_allK:
        contrast_allk = pd.concat(all_for_contrast_allK, ignore_index=True)
        save_bar_contrast_with_k(
            df_long=contrast_allk[["Dataset","mode","index","ARI"]],
            T_meta=out["allk"][["Dataset","mode","index","kA","kB","k_max"]] if "allk" in out else pd.DataFrame(),
            out_dir=OUT_APPENDIX,
            title="Appendix: OH vs FP (ARI of new vs signless, all k)",
            fname_core="Appendix_Fig_allK_OHFP-contrast_ARI-bars",
            fade_outside=True
        )

    return out

# ========= 逆方向（FP→OH） with FPラベル =========
FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)
def latest_by_mode_index(fig_dir: Path, ds: str):
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds: 
            continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k: v[1] for k,v in latest.items()}

def read_var_cluster(path: Path) -> pd.Series:
    df = read_csv_quiet(path)
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try: s = s.astype(int)
    except: pass
    return s

def entropy_from_dist(dist: dict[int, float]) -> float:
    if not dist: return np.nan
    p = np.array(list(dist.values()), dtype=float)
    p = p / (p.sum() + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def js_divergence(p, q):
    p = np.asarray(p, dtype=float); p = p/p.sum() if p.sum()>0 else p
    q = np.asarray(q, dtype=float); q = q/q.sum() if q.sum()>0 else q
    m = 0.5*(p+q)
    def _kl(a,b):
        a = np.clip(a,1e-12,1); b = np.clip(b,1e-12,1)
        return float(np.sum(a*(np.log2(a)-np.log2(b))))
    return 0.5*_kl(p,m) + 0.5*_kl(q,m)

def build_reverse_direction() -> pd.DataFrame:
    """
    逆方向の 3分析表 + summary を出力し、summary（all-k, kOH/kFP/k_max 付き）を返す。
    """
    set_font()
    Xoh0 = load_features_bin(FEATURES_OH)
    Xfp0 = load_features_bin(FEATURES_FP)
    common = Xoh0.index.intersection(Xfp0.index)
    if len(common)==0: 
        raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
    Xoh0 = Xoh0.loc[common]; Xfp0 = Xfp0.loc[common]
    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
    if not keys: 
        raise RuntimeError("No common (mode,index) between figs_OH and figs_FP.")

    summary_rows=[]
    for (mode,index) in keys:
        print(f"[REV] {mode} × {index}")
        oh_assign = read_var_cluster(oh_latest[(mode,index)])
        fp_assign = read_var_cluster(fp_latest[(mode,index)])

        # kOH / kFP
        kOH = int(pd.Series(oh_assign).dropna().nunique()) if len(oh_assign) else np.nan
        kFP = int(pd.Series(fp_assign).dropna().nunique()) if len(fp_assign) else np.nan
        k_max = np.nanmax([kOH, kFP]) if (np.isfinite(kOH) or np.isfinite(kFP)) else np.nan

        # ---- フラグメント → OH 主分布 ----
        fragments = [f for f in fp_assign.index if f in Xfp0.columns]
        rows_frag=[]
        for frag in fragments:
            idx = Xfp0.index[Xfp0[frag]==1]
            n_frag = int(len(idx))
            if n_frag<=0: 
                continue
            prev = Xoh0.loc[idx].mean(axis=0)
            sel = prev[prev>0]
            if sel.empty:
                maj, purity, dist = None, None, {}
            else:
                grp = sel.sort_values(ascending=False)
                maj = grp.index[0]
                purity = float(grp.iloc[0] / grp.sum())
                dist = dict(grp / grp.sum())

            fp_c = fp_assign.get(frag, np.nan)
            H = entropy_from_dist(dist)
            k_eff = len(dist) if dist else 0

            rows_frag.append({
                "mode": mode, "index": index,
                "fragment": frag,
                "fragment_label": fp_var_display_name(frag),
                "FP_cluster": int(fp_c) if pd.notna(fp_c) else np.nan,
                "n_samples_with_fragment": n_frag,
                "OH_major_material": maj,
                "OH_purity": purity,
                "OH_entropy": H,
                "OH_k_effective": k_eff,
                "OH_dist": ";".join(f"{k}:{v:.3f}" for k,v in sorted(dist.items())) if dist else ""
            })

        df_frag = pd.DataFrame(rows_frag)
        out_csv1 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv"
        if len(df_frag): 
            df_frag.to_csv(out_csv1, index=False, encoding="utf-8-sig")
            print(f"[SAVE] {out_csv1}")

        # ---- ペア ----
        rows_pair=[]
        if len(df_frag):
            def parse_dist(s):
                if isinstance(s,str) and s:
                    d={}
                    for t in s.split(";"):
                        k,v = t.split(":"); d[str(k)] = float(v)
                    return d
                return {}
            dists = {r.fragment: parse_dist(r.OH_dist) for r in df_frag.itertuples()}
            all_mats = sorted({k for d in dists.values() for k in d.keys()})
            def vec_of(dist):
                return np.array([dist.get(k, 0.0) for k in all_mats], dtype=float)

            for k, grp in df_frag.dropna(subset=["FP_cluster"]).groupby("FP_cluster"):
                frags = grp["fragment"].tolist()
                for a,b in combinations(frags,2):
                    da, db = dists.get(a,{}), dists.get(b,{})
                    va, vb = vec_of(da), vec_of(db)
                    if va.sum()==0 and vb.sum()==0:
                        cos_sim=np.nan; jsd=np.nan
                    else:
                        cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
                        jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
                    maj_a = df_frag.loc[df_frag["fragment"]==a, "OH_major_material"].values[0]
                    maj_b = df_frag.loc[df_frag["fragment"]==b, "OH_major_material"].values[0]
                    maj_same = (maj_a == maj_b)
                    purity_min = float(min(
                        df_frag.loc[df_frag["fragment"]==a, "OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==a, "OH_purity"]) else 0.0,
                        df_frag.loc[df_frag["fragment"]==b, "OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==b, "OH_purity"]) else 0.0
                    ))
                    rows_pair.append({
                        "mode":mode,"index":index,
                        "FP_cluster":int(k),
                        "fragment_A":a,
                        "fragment_A_label": fp_var_display_name(a),
                        "fragment_B":b,
                        "fragment_B_label": fp_var_display_name(b),
                        "OH_major_same": bool(maj_same),
                        "OH_purity_min": purity_min,
                        "OH_cosine_similarity": cos_sim,
                        "OH_JS_divergence": jsd
                    })
        df_pair = pd.DataFrame(rows_pair)
        out_csv2 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv"
        if len(df_pair): 
            df_pair.to_csv(out_csv2, index=False, encoding="utf-8-sig")
            print(f"[SAVE] {out_csv2}")

        # ---- FPクラスタ単位の材料側コヒージョン ----
        if len(df_pair):
            def label_cohesion(r, purity_thr=0.60, cos_thr=0.60, js_thr=0.50):
                return bool(
                    (r["OH_major_same"] and r["OH_purity_min"]>=purity_thr) or
                    (pd.notna(r["OH_cosine_similarity"]) and r["OH_cosine_similarity"]>=cos_thr) or
                    (pd.notna(r["OH_JS_divergence"]) and r["OH_JS_divergence"]<=js_thr)
                )
            df_pair["cohesive_flag"] = df_pair.apply(label_cohesion, axis=1)
            df_fpc = (df_pair.groupby(["mode","index","FP_cluster"], as_index=False)
                             .agg(n_pairs=("cohesive_flag","size"),
                                  n_cohesive=("cohesive_flag","sum")))
            df_fpc["cohesive_ratio"] = df_fpc["n_cohesive"]/df_fpc["n_pairs"].replace(0,np.nan)
        else:
            df_fpc = pd.DataFrame()
        out_csv3 = OUT_R_CSV / f"Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv"
        if len(df_fpc): 
            df_fpc.to_csv(out_csv3, index=False, encoding="utf-8-sig")
            print(f"[SAVE] {out_csv3}")

        # ---- 図（本文/付録） ----
        is_main = (mode,index) in MAIN_PAIRS
        out_dir = (OUT_R_MAIN if is_main else OUT_R_APPENDIX) / f"FigSet_4.2.4.4R_{mode}_{index}"
        out_dir.mkdir(parents=True, exist_ok=True)
        plot_fragment_scatter(df_frag, out_dir, mode, index, top_labels=15, main_text=is_main)
        plot_pair_distributions_reverse(df_pair, out_dir, mode, index, main_text=is_main)
        plot_fpcluster_consistency(df_fpc, out_dir, mode, index, main_text=is_main)

        # ---- summary 行（kOH/kFP/k_max を含める） ----
        summary_rows.append({
            "mode":mode,"index":index,
            "kOH": kOH, "kFP": kFP, "k_max": k_max,
            "n_fragments": len(df_frag),
            "mean_OH_purity": df_frag["OH_purity"].dropna().mean() if "OH_purity" in df_frag else np.nan,
            "mean_OH_entropy": df_frag["OH_entropy"].dropna().mean() if "OH_entropy" in df_frag else np.nan,
            "pair_OH_major_same_rate": df_pair["OH_major_same"].mean() if len(df_pair) else np.nan,
            "pair_mean_cosine_similarity": df_pair["OH_cosine_similarity"].dropna().mean() if len(df_pair) else np.nan,
            "pair_mean_JS_divergence": df_pair["OH_JS_divergence"].dropna().mean() if len(df_pair) else np.nan,
            "FPcluster_median_cohesive_ratio": df_fpc["cohesive_ratio"].median() if len(df_fpc) else np.nan
        })

    outsum = OUT_REVERSE_ROOT / "Table_4.2.4.4R_summary_all_conditions.csv"
    Rsum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
    Rsum.to_csv(outsum, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {outsum}")
    return Rsum

# ========= 図（逆方向; FPラベルを使用） =========
def plot_fragment_scatter(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
    set_font()
    if df_frag.empty: return
    size = 30 + 3 * df_frag["n_samples_with_fragment"].fillna(0).astype(float)
    c    = df_frag.get("FP_cluster", pd.Series(np.nan, index=df_frag.index)).astype("category")
    fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
    ax.scatter(df_frag["OH_entropy"], df_frag["OH_purity"], c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
    ax.set_xlabel("OH entropy (larger = more materials-spread)")
    ax.set_ylabel("OH purity (larger = more materials-specific)")
    title = f"Fig 4.2.4.4R: Fragments — Specificity vs Spread ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    lab_df = df_frag.sort_values(["OH_purity","n_samples_with_fragment"], ascending=False).head(top_labels)
    for _, r in lab_df.iterrows():
        lbl = str(r.get("fragment_label", r["fragment"]))
        ax.text(r["OH_entropy"], r["OH_purity"]+0.015, lbl, ha="center", va="bottom", fontsize=8)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_fragments_specificity-vs-spread_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_pair_distributions_reverse(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_pair is None or df_pair.empty: return
    set_font()
    # major same
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    rate = df_pair["OH_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
    idx = ["Same","Different"]; vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
    ax.bar(idx, vals, alpha=0.9)
    ax.set_ylim(0,1); ax.set_ylabel("Proportion")
    title = f"Fig 4.2.4.4R: Pair-level — OH major same? ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, val in enumerate(vals): ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_pair_major-same_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)
    # cosine / JS
    for col, xlabel in [("OH_cosine_similarity","Cosine similarity (1=similar)"),
                        ("OH_JS_divergence","JS divergence (0=similar)")]:
        s = df_pair[col].dropna()
        if not len(s): continue
        fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
        ax.hist(s, bins=20, alpha=0.9)
        title = f"Fig 4.2.4.4R: Pair-level — {col} ({mode} × {index})"
        if main_text: title += " [Main]"
        ax.set_title(title)
        ax.set_xlabel(xlabel); ax.set_ylabel("Count")
        fig.tight_layout()
        fname = f"Fig_4.2.4.4R_pair_{col}_hist_{mode}_{index}{'_MAIN' if main_text else ''}"
        fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
        plt.close(fig)

def plot_fpcluster_consistency(df_fpc: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_fpc is None or df_fpc.empty: return
    set_font()
    df = df_fpc.sort_values("cohesive_ratio", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    x = np.arange(len(df))
    ax.bar(x, df["cohesive_ratio"].values, alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels([f"FP {int(c)}" for c in df["FP_cluster"]], rotation=35, ha="right")
    ax.set_ylim(0,1.05)
    ax.set_ylabel("Cohesive ratio (pair-level in OH space)")
    title = f"Fig 4.2.4.4R: FP-cluster cohesion in OH space ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, v in zip(x, df["cohesive_ratio"].values): ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_FPcluster_cohesion_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

# ========= reverse / forward から bidirectional (レガシー棒) を作る =========
def _minmax_series(s: pd.Series) -> pd.Series:
    s2 = s.copy()
    if s2.isna().all(): return pd.Series(0.5, index=s2.index)
    s2 = s2.fillna(s2.mean())
    mn, mx = s2.min(), s2.max()
    if not np.isfinite(mn) or not np.isfinite(mx) or np.isclose(mx, mn):
        return pd.Series(0.5, index=s2.index)
    return (s2 - mn) / (mx - mn)

def make_bidirectional_summary(forward_df: pd.DataFrame, reverse_df: pd.DataFrame,
                               k_mode: str, k_threshold: int = K_MAX_KEEP_DEFAULT) -> pd.DataFrame:
    """
    forward_df: forward の行明細（OH→FP; ARI を持つ）
    reverse_df: reverse summary（kOH/kFP/k_max 付き）

    k_mode:
      - "legacy" : forward は k≤threshold 版を渡す / reverse は all-k の reverse_df をそのまま
      - "allk"   : 両方 all-k; forward_df は allk を渡す
      - "unify30": 両方 k≤threshold; reverse も k_max でフィルタ
    """
    # forward_score
    fwd_sc = (forward_df.groupby(["mode","index"])["ARI"].mean().rename("forward_score")).reset_index() \
             if ("ARI" in forward_df.columns and len(forward_df)) else \
             pd.DataFrame(columns=["mode","index","forward_score"])

    # reverse_score の素材
    rev = reverse_df.copy()
    # k-mode に応じて filter
    if k_mode == "unify30":
        rev = rev[(rev["k_max"].isna()) | (rev["k_max"]<=k_threshold)].copy()

    # composite
    pos = [c for c in ["mean_OH_purity","pair_OH_major_same_rate","pair_mean_cosine_similarity","FPcluster_median_cohesive_ratio"] if c in rev.columns]
    neg = [c for c in ["mean_OH_entropy","pair_mean_JS_divergence"] if c in rev.columns]
    parts=[]
    for c in pos: parts.append(_minmax_series(rev[c]))
    for c in neg: parts.append(1 - _minmax_series(rev[c]))
    rev["reverse_score"] = pd.concat(parts, axis=1).mean(axis=1) if parts else np.nan
    rev_sc = rev.groupby(["mode","index"])["reverse_score"].mean().reset_index()

    M = fwd_sc.merge(rev_sc, on=["mode","index"], how="outer").sort_values(["mode","index"])
    out_csv = OUT_BIDIR / "Table_4.2.4.4_bidirectional_summary.csv"
    M.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out_csv}")

    # 図
    set_font()
    fig, ax = plt.subplots(figsize=(10,5), dpi=DPI)
    lbls = [f"{m}-{i}" for m,i in zip(M["mode"], M["index"])]
    x = np.arange(len(M)); w = 0.38
    ax.bar(x-w/2, M["forward_score"].values, width=w, label="OH→FP (ARI mean)", alpha=0.9)
    ax.bar(x+w/2, M["reverse_score"].values, width=w, label="FP→OH (composite)", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("Score (0–1)")
    ax.set_title(f"Fig 4.2.4.4: Bidirectional alignment summary  [{k_mode}]")
    ax.legend()
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.png", bbox_inches="tight")
    fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.pdf", bbox_inches="tight")
    plt.close(fig)
    return M

# ========= 追加：結合（joined）→ 相関 → 図 → サマリー・統計（BH） =========
def _ensure_origin(df: pd.DataFrame) -> pd.DataFrame:
    if "origin" not in df.columns:
        df["origin"] = "unknown"
    return df

def _bh_adjust(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    if _HAS_STATSMODELS:
        mask = np.isfinite(p)
        out = np.full_like(p, np.nan)
        if mask.any():
            _, pa, _, _ = multipletests(p[mask], method="fdr_bh")
            out[mask] = pa
        return out
    mask = np.isfinite(p)
    out = np.full_like(p, np.nan)
    vals = p[mask]
    if len(vals)==0: return out
    order = np.argsort(vals)
    m = float(len(vals))
    adj = np.empty_like(vals)
    for r, idx in enumerate(order, start=1):
        adj[idx] = vals[idx] * m / r
    for i in range(len(adj)-2, -1, -1):
        adj[i] = min(adj[i], adj[i+1])
    adj = np.clip(adj, 0, 1)
    out[mask] = adj
    return out

def _pearson_spearman_long(df: pd.DataFrame, out_csv: Path):
    if df.empty or df.shape[1]<2:
        pd.DataFrame(columns=["method","index","variable","value"]).to_csv(out_csv, index=False, encoding="utf-8-sig")
        return {"pearson": pd.DataFrame(), "spearman": pd.DataFrame()}
    d = df.dropna(how="any")
    if d.shape[0] < 2:
        pd.DataFrame(columns=["method","index","variable","value"]).to_csv(out_csv, index=False, encoding="utf-8-sig")
        return {"pearson": pd.DataFrame(), "spearman": pd.DataFrame()}
    pear = d.corr(method="pearson")
    spear = d.corr(method="spearman")
    pear_long = pear.reset_index().melt(id_vars="index", var_name="variable", value_name="value"); pear_long.insert(0,"method","pearson")
    spear_long= spear.reset_index().melt(id_vars="index", var_name="variable", value_name="value"); spear_long.insert(0,"method","spearman")
    corr_long = pd.concat([pear_long, spear_long], ignore_index=True)
    corr_long.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return {"pearson": pear, "spearman": spear}

def build_joined_cor_plot_stats(forward_df: pd.DataFrame, reverse_df: pd.DataFrame, k_mode: str) -> None:
    """
    forward_df : 使用する forward 明細（k-mode に応じて allk or k30）
    reverse_df : 逆方向 summary（必要に応じて k_max でサブセット）
    """
    # forward wide（ARI_mean 等）
    A = forward_df.copy()
    keep = ["mode","index","Dataset","ARI","NMI","AMI","kA","kB","n"]
    for c in keep:
        if c not in A.columns: A[c] = np.nan
    w = (A[keep].pivot_table(index=["mode","index"], columns="Dataset",
                                values=["ARI","NMI","AMI","kA","kB","n"], aggfunc="first"))
    w.columns = [f"{a}_{b}" for a,b in w.columns]
    w = w.reset_index()
    w = w.rename(columns={c: c.replace("_oh","_OH").replace("_fp","_FP") for c in w.columns})
    def mean2(a,b):
        a = pd.to_numeric(a, errors="coerce"); b = pd.to_numeric(b, errors="coerce")
        return pd.concat([a,b], axis=1).mean(axis=1, skipna=True)
    if {"ARI_OH","ARI_FP"}.issubset(w.columns): w["ARI_mean"] = mean2(w["ARI_OH"], w["ARI_FP"])
    if {"NMI_OH","NMI_FP"}.issubset(w.columns): w["NMI_mean"] = mean2(w["NMI_OH"], w["NMI_FP"])
    if {"AMI_OH","AMI_FP"}.issubset(w.columns): w["AMI_mean"] = mean2(w["AMI_OH"], w["AMI_FP"])

    # reverse（k-mode に応じて制御）
    R = reverse_df.copy()
    if k_mode == "unify30":
        R = R[(R["k_max"].isna()) | (R["k_max"]<=K_MAX_KEEP_DEFAULT)].copy()

    # 結合
    M = (w.merge(R, on=["mode","index"], how="outer")
           .sort_values(["mode","index"]).reset_index(drop=True))
    M = _ensure_origin(M)

    # JS (rev)
    if "pair_mean_JS_divergence" in M.columns:
        js = pd.to_numeric(M["pair_mean_JS_divergence"], errors="coerce")
        if js.notna().any():
            if (js.min()>=0) and (js.max()<=1):
                M["JS_divergence_rev"] = 1 - js
            else:
                mn, mx = js.min(), js.max()
                if np.isfinite(mn) and np.isfinite(mx) and mx>mn:
                    M["JS_divergence_rev"] = 1 - (js - mn) / (mx - mn)
                else:
                    M["JS_divergence_rev"] = np.nan

    # 保存：joined
    joined_csv = OUT_BIDIR / "Table_4.2.4.4_bidirectional_joined.csv"
    M.to_csv(joined_csv, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {joined_csv}")

    # 相関
    cols_interest = [c for c in [
        "ARI_mean","NMI_mean","AMI_mean",
        "pair_OH_major_same_rate","pair_mean_cosine_similarity","pair_mean_JS_divergence","JS_divergence_rev",
        "FPcluster_median_cohesive_ratio","mean_OH_purity","mean_OH_entropy"
    ] if c in M.columns]
    Dnum = M[cols_interest].apply(pd.to_numeric, errors="coerce")
    corr_csv = OUT_BIDIR / "Table_4.2.4.4_bidirectional_correlations.csv"
    _ = _pearson_spearman_long(Dnum, corr_csv)
    print(f"[SAVE] {corr_csv}")

    # 図（散布）
    def save_scatter(xcol, ycol, ttl_suffix):
        if (xcol not in M.columns) or (ycol not in M.columns): return
        x = pd.to_numeric(M[xcol], errors="coerce")
        y = pd.to_numeric(M[ycol], errors="coerce")
        mask = x.notna() & y.notna()
        n = int(mask.sum())
        if n < 1: return
        set_font()
        fig, ax = plt.subplots(figsize=(6,4), dpi=DPI)
        ax.scatter(x[mask].values, y[mask].values, alpha=0.9)
        if n >= 2:
            coef = np.polyfit(x[mask].values, y[mask].values, 1)
            xs = np.linspace(float(np.min(x[mask].values)), float(np.max(x[mask].values)), 100)
            ys = np.polyval(coef, xs)
            ax.plot(xs, ys)
        ax.set_xlabel(xcol); ax.set_ylabel(ycol); ax.set_title(f"Fig 4.2.4.4: {ttl_suffix} (n={n})")
        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
        fig.tight_layout()
        fig.savefig(OUT_BIDIR / f"Fig_4.2.4.4_bidirectional_ARIscatter_{xcol}_vs_{ycol}.png", bbox_inches="tight")
        fig.savefig(OUT_BIDIR / f"Fig_4.2.4.4_bidirectional_ARIscatter_{xcol}_vs_{ycol}.pdf", bbox_inches="tight")
        plt.close(fig)

    for xcol, ycol, ttl in [
        ("ARI_mean","pair_OH_major_same_rate","ARI_mean vs pair_OH_major_same_rate"),
        ("ARI_mean","pair_mean_cosine_similarity","ARI_mean vs pair_mean_cosine_similarity"),
        ("ARI_mean","FPcluster_median_cohesive_ratio","ARI_mean vs FPcluster_median_cohesive_ratio"),
        ("ARI_mean","JS_divergence_rev","ARI_mean vs JS_divergence_rev"),
    ]:
        save_scatter(xcol, ycol, ttl)

    # 箱ひげ（origin）
    def save_box(metric, label):
        if metric not in M.columns: return
        d = pd.DataFrame({"origin": M["origin"], metric: pd.to_numeric(M[metric], errors="coerce")}).dropna()
        if d.empty: return
        set_font()
        fig, ax = plt.subplots(figsize=(7,5), dpi=DPI)
        groups = [g[metric].values for _, g in d.groupby("origin")]
        labels = [str(k) for k,_ in d.groupby("origin")]
        ax.boxplot(groups, labels=labels, showfliers=False)
        ax.set_xlabel("origin"); ax.set_ylabel(label); ax.set_title(f"{label} by origin")
        ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
        fig.tight_layout()
        core = re.sub(r"[^A-Za-z0-9_]+","_", metric)
        fig.savefig(OUT_BIDIR / f"Fig_4.2.4.4_bidirectional_box_{core}.png", bbox_inches="tight")
        fig.savefig(OUT_BIDIR / f"Fig_4.2.4.4_bidirectional_box_{core}.pdf", bbox_inches="tight")
        plt.close(fig)

    for m, label in [
        ("ARI_mean","ARI (OH→FP)"),
        ("pair_mean_cosine_similarity","Pair cosine similarity (FP→OH)"),
        ("FPcluster_median_cohesive_ratio","FP cluster cohesion (FP→OH)"),
        ("JS_divergence_rev","Pair JS divergence (rev., higher=better; FP→OH)")
    ]:
        save_box(m, label)

    # 由来別サマリー
    metrics_for_summary = [m for m in ["ARI_mean","pair_mean_cosine_similarity","FPcluster_median_cohesive_ratio","JS_divergence_rev"] if m in M.columns]
    if metrics_for_summary:
        g = M.groupby("origin")
        S = g[metrics_for_summary].agg(["mean","std","count"])
        S.columns = [f"{m}_{s}" for m,s in S.columns]
        S = S.reset_index()
        S["n_runs"] = g.size().values
        sum_csv = OUT_BIDIR / "Table_4.2.4.4_bidirectional_summary.csv"
        S.to_csv(sum_csv, index=False, encoding="utf-8-sig")
        print(f"[SAVE] {sum_csv}")

    # 統計（z差）
    def zscore(s):
        s = pd.to_numeric(s, errors="coerce")
        sd = s.std(ddof=1)
        if not np.isfinite(sd) or sd==0: return s*0.0
        return (s - s.mean())/sd

    rows=[]
    if "ARI_mean" in M.columns:
        for org, sub in _ensure_origin(M).groupby("origin"):
            z = sub.copy()
            z["ARI_mean_z"] = zscore(z["ARI_mean"])
            for m in metrics_for_summary:
                if m == "ARI_mean" or m not in sub.columns: continue
                z[m+"_z"] = zscore(z[m])
                diff = (z["ARI_mean_z"] - z[m+"_z"]).dropna()
                if diff.empty: continue
                if len(diff) >= 2:
                    t1, p1 = stats.ttest_1samp(diff, popmean=0.0, nan_policy="omit")
                else:
                    t1, p1 = (np.nan, np.nan)
                # Welch raw
                x = pd.to_numeric(sub["ARI_mean"], errors="coerce").dropna()
                y = pd.to_numeric(sub[m], errors="coerce").dropna()
                if len(x)>=2 and len(y)>=2:
                    t2, p2 = stats.ttest_ind(x, y, equal_var=False, nan_policy="omit")
                    nx, ny = x.count(), y.count()
                    sx, sy = x.std(ddof=1), y.std(ddof=1)
                    if nx>=2 and ny>=2 and np.isfinite(sx) and np.isfinite(sy) and (nx+ny-2)>0:
                        sp = math.sqrt(((nx-1)*sx*sx + (ny-1)*sy*sy)/(nx+ny-2))
                        d2 = (x.mean() - y.mean())/sp if (sp and np.isfinite(sp)) else np.nan
                    else:
                        d2 = np.nan
                else:
                    t2, p2, d2 = (np.nan, np.nan, np.nan)

                rows.append({
                    "origin": org, "metric_FPtoOH": m,
                    "n_diff": int(diff.count()),
                    "mean_diff_z": float(diff.mean()),
                    "t_one_sample": float(t1) if np.isfinite(t1) else np.nan,
                    "p_one_sample": float(p1) if np.isfinite(p1) else np.nan,
                    "t_welch_raw": float(t2) if np.isfinite(t2) else np.nan,
                    "p_welch_raw": float(p2) if np.isfinite(p2) else np.nan,
                    "cohens_d_ind_raw": float(d2) if np.isfinite(d2) else np.nan,
                    "mean_OHtoFP_raw": float(x.mean()) if len(x) else np.nan,
                    f"mean_{m}_raw": float(y.mean()) if len(y) else np.nan
                })

    stats_df = pd.DataFrame(rows)
    if not stats_df.empty:
        stats_df["p_BH_one_sample"] = np.nan
        stats_df["p_BH_welch_raw"] = np.nan
        for m in stats_df["metric_FPtoOH"].unique():
            mask = stats_df["metric_FPtoOH"] == m
            stats_df.loc[mask, "p_BH_one_sample"] = _bh_adjust(stats_df.loc[mask, "p_one_sample"].to_numpy())
            stats_df.loc[mask, "p_BH_welch_raw"]  = _bh_adjust(stats_df.loc[mask, "p_welch_raw"].to_numpy())
        out_stats = OUT_BIDIR / "Table_4.2.4.4_bidirectional_stats.csv"
        stats_df.to_csv(out_stats, index=False, encoding="utf-8-sig")
        print(f"[SAVE] {out_stats}")

    # README
    readme = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "k_mode": k_mode,
        "outputs": {
            "joined": str(OUT_BIDIR / "Table_4.2.4.4_bidirectional_joined.csv"),
            "correlations": str(OUT_BIDIR / "Table_4.2.4.4_bidirectional_correlations.csv"),
            "summary_by_origin": str(OUT_BIDIR / "Table_4.2.4.4_bidirectional_summary.csv"),
            "stats": str(OUT_BIDIR / "Table_4.2.4.4_bidirectional_stats.csv"),
            "figs_scatter": sorted([p.name for p in OUT_BIDIR.glob("Fig_4.2.4.4_bidirectional_ARIscatter_*.png")]),
            "figs_box": sorted([p.name for p in OUT_BIDIR.glob("Fig_4.2.4.4_bidirectional_box_*.png")]),
            "legacy_bar": str(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.pdf")
        }
    }
    (OUT_BIDIR / "README_bidirectional_summary.json").write_text(
        json.dumps(readme, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"[SAVE] {OUT_BIDIR / 'README_bidirectional_summary.json'}")

# ========= CLI & MAIN =========
def main(k_mode: str, run_forward=True, run_reverse=True, run_legacy_bidir=True, run_joined_block=True, run_validation=False):
    """
    k_mode: "legacy" / "allk" / "unify30"
    """
    set_font()

    # 1) forward（本文表）を作成（all-k と k≤30 の両方作る）
    print("=== [1/4] OH→FP contrast (new vs signless) ===")
    fwd_tables = build_ohfp_contrast_tables()
    F_allk = fwd_tables.get("allk", pd.DataFrame())
    F_k30  = fwd_tables.get("k30",  pd.DataFrame())

    # 2) reverse（FP→OH）集計（kOH/kFP/k_max 付き）
    print("=== [2/4] FP→OH reverse direction (with FP labels & k) ===")
    Rsum = build_reverse_direction()  # all-k summary

    # 3) bidirectional（レガシー棒）— k-mode に応じて forward/reverse を選択
    print("=== [3/4] Bidirectional summary (legacy bar summary) ===")
    if k_mode == "legacy":
        forward_df = F_k30
        reverse_df = Rsum                 # all-k
    elif k_mode == "allk":
        forward_df = F_allk
        reverse_df = Rsum                 # all-k
    elif k_mode == "unify30":
        forward_df = F_k30
        reverse_df = Rsum                 # 後段で k_max ≤ 30 に制限
    else:
        raise ValueError(f"Unknown k_mode: {k_mode}")

    make_bidirectional_summary(forward_df=forward_df, reverse_df=reverse_df, k_mode=k_mode)

    # 4) Joined → Corr → Plots → Summary/Stats（選んだ forward/reverse で）
    print("=== [4/4] Joined → Corr → Plots → Summary/Stats (BH) ===")
    build_joined_cor_plot_stats(forward_df=forward_df, reverse_df=reverse_df, k_mode=k_mode)

    print("\n✅ Done. Outputs under:", OUT_OHFP_ROOT)

if __name__ == "__main__":
    warnings.filterwarnings("ignore")
    parser = argparse.ArgumentParser(description="4.2.4.4 build-all (FP labels; k-mode selectable)")
    parser.add_argument("--k-mode", type=str, default=DEFAULT_K_MODE,
                        choices=["legacy","allk","unify30"],
                        help="k 取り扱いの方針: legacy(既定)/allk/unify30")
    parser.add_argument("--no-forward", action="store_true", help="skip OH→FP contrast tables (not recommended)")
    parser.add_argument("--no-reverse", action="store_true", help="skip FP→OH reverse (not recommended)")
    parser.add_argument("--no-legacy-bidir", action="store_true", help="skip legacy bidirectional bar")
    parser.add_argument("--no-joined-block", action="store_true", help="skip Joined→Corr→Plots→Summary/Stats")
    parser.add_argument("--validation", action="store_true", help="(placeholder) run validation")
    parser.add_argument("--root", type=str, default=None, help="ROOT を上書きする（ベースディレクトリ）")
    args, _ = parser.parse_known_args()

    if args.root:
        global ROOT, FIGS_OH, FIGS_FP, SIGNLESS_BASE, SIGNLESS_DIR, FEATURES_OH, FEATURES_FP
        ROOT = Path(args.root).resolve()
        FIGS_OH = ROOT / "figs_OH"
        FIGS_FP = ROOT / "figs_FP"
        SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"
        SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}
        FEATURES_OH = ROOT / "features_OH_onlyMat.csv"
        FEATURES_FP = ROOT / "features_FP_onlyMat.csv"

        # 出力ルートも更新
        global OUT_OHFP_ROOT, OUT_MAIN, OUT_APPENDIX, OUT_REVERSE_ROOT, OUT_R_MAIN, OUT_R_APPENDIX, OUT_R_CSV, OUT_BIDIR
        OUT_OHFP_ROOT     = ROOT / "paper_4.2.4.4_OHFP"
        OUT_MAIN          = OUT_OHFP_ROOT / "main_text"
        OUT_APPENDIX      = OUT_OHFP_ROOT / "appendix"
        OUT_REVERSE_ROOT  = OUT_OHFP_ROOT / "reverse"
        OUT_R_MAIN        = OUT_REVERSE_ROOT / "main_text"
        OUT_R_APPENDIX    = OUT_REVERSE_ROOT / "appendix"
        OUT_R_CSV         = OUT_REVERSE_ROOT / "analysis_csv"
        OUT_BIDIR         = OUT_OHFP_ROOT / "bidirectional_summary"
        for d in [OUT_MAIN, OUT_APPENDIX, OUT_R_MAIN, OUT_R_APPENDIX, OUT_R_CSV, OUT_BIDIR]:
            d.mkdir(parents=True, exist_ok=True)

    k_mode = args.k_mode or DEFAULT_K_MODE

    main(k_mode=k_mode,
         run_forward=not args.no_forward,
         run_reverse=not args.no_reverse,
         run_legacy_bidir=not args.no_legacy_bidir,
         run_joined_block=not args.no_joined_block,
         run_validation=bool(args.validation))

=== [1/4] OH→FP contrast (new vs signless) ===
[WARN] OH: no common conditions
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/appendix/Appendix_Table_allK_OHFP-contrast_summary.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/main_text/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv
=== [2/4] FP→OH reverse direction (with FP labels & k) ===
[REV] top3 × silhouette
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_fragment-to-OHmajor_top3_silhouette.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_fragment-pair_cohesion_top3_silhouette.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_FPcluster_cohesion_top3_silhouette.csv
[REV] top3 × dunn
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebui